In [0]:
%pip install "PyJWT[crypto]" requests

In [0]:
import json
import time

import jwt
import requests

APP_ID = "2330c837-db68-4541-b14e-19d56818fcb6"

private_key = dbutils.secrets.get(
    "openbanking",
    "enablebanking-private-key",
)

now = int(time.time())

token = jwt.encode(
    {
        "iss": "enablebanking.com",
        "aud": "api.enablebanking.com",
        "iat": now,
        "exp": now + 3600,
    },
    private_key,
    algorithm="RS256",
    headers={"kid": APP_ID},
)

response = requests.get(
    "https://api.enablebanking.com/application",
    headers={"Authorization": f"Bearer {token}"},
    timeout=30,
)

print("HTTP:", response.status_code)
response.raise_for_status()

application = response.json()
print("Entorno:", application.get("environment"))
print("Aplicación activa:", application.get("active"))

for bank in ("santander", "laboral"):
    session = json.loads(
        dbutils.secrets.get("openbanking", f"session-{bank}")
    )
    print(f"{bank}: {len(session.get('accounts', []))} cuentas en la sesión")

In [0]:
for bank in ("santander", "laboral"):
    session = json.loads(
        dbutils.secrets.get("openbanking", f"session-{bank}")
    )

    for account in session.get("accounts", []):
        response = requests.get(
            (
                "https://api.enablebanking.com"
                f"/accounts/{account['uid']}/balances"
            ),
            headers={"Authorization": f"Bearer {token}"},
            timeout=60,
        )

        print(f"\nBanco: {bank}")
        print("HTTP:", response.status_code)

        if not response.ok:
            # Mostrar solo el tipo de error, sin datos de cuenta.
            try:
                print("Error:", response.json().get("error"))
            except ValueError:
                print("La respuesta no contiene JSON.")
            response.raise_for_status()

        print(
            "Saldos recibidos:",
            len(response.json().get("balances", [])),
        )

In [0]:
from datetime import datetime, timedelta, timezone

now = int(time.time())

token = jwt.encode(
    {
        "iss": "enablebanking.com",
        "aud": "api.enablebanking.com",
        "iat": now,
        "exp": now + 3600,
    },
    private_key,
    algorithm="RS256",
    headers={"kid": APP_ID},
)

session = json.loads(
    dbutils.secrets.get("openbanking", "session-laboral")
)

today = datetime.now(timezone.utc).date()

for account in session.get("accounts", []):
    response = requests.get(
        (
            "https://api.enablebanking.com"
            f"/accounts/{account['uid']}/transactions"
        ),
        headers={"Authorization": f"Bearer {token}"},
        params={
            "date_from": (today - timedelta(days=7)).isoformat(),
            "date_to": today.isoformat(),
        },
        timeout=60,
    )

    print("Banco: laboral")
    print("HTTP:", response.status_code)

    payload = response.json()

    if response.ok:
        print(
            "Transacciones en esta página:",
            len(payload.get("transactions", [])),
        )
        print(
            "Hay más páginas:",
            bool(payload.get("continuation_key")),
        )
    else:
        print("Error:", payload.get("error"))
        print("Mensaje:", payload.get("message"))

In [0]:
%pip install "PyJWT[crypto]" requests

In [0]:
import json
import sys
from pathlib import Path

# Busca la raíz del proyecto desde la ubicación del notebook.
current = Path.cwd()
project_root = next(
    (
        path
        for path in (current, *current.parents)
        if (path / "config/banking.json").is_file()
        and (path / "src/openbanking").is_dir()
    ),
    None,
)

if project_root is None:
    raise RuntimeError("No se ha encontrado la raíz del proyecto.")

source_path = str(project_root / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

from openbanking.enablebanking_client import EnableBankingClient

config = json.loads(
    (project_root / "config/banking.json").read_text(encoding="utf-8")
)

print("Módulo y configuración cargados correctamente.")

In [0]:
from datetime import datetime, timedelta, timezone

date_to = datetime.now(timezone.utc).date()
date_from = date_to - timedelta(days=6)

private_key = dbutils.secrets.get(
    config["secret_scope"],
    config["private_key_secret"],
)

print(f"Rango solicitado: {date_from} a {date_to} (UTC)")

with EnableBankingClient(
    application_id=config["application_id"],
    private_key=private_key,
    base_url=config["base_url"],
    timeout=config["extraction"]["request_timeout_seconds"],
) as client:

    for bank, bank_config in config["banks"].items():
        session = json.loads(
            dbutils.secrets.get(
                config["secret_scope"],
                bank_config["session_secret"],
            )
        )

        accounts = session.get("accounts", [])
        if not accounts:
            raise RuntimeError(f"{bank}: sesión sin cuentas.")

        print(f"\nBanco: {bank}")

        for index, account in enumerate(accounts, start=1):
            balances = client.get_balances(account["uid"])

            page_count = 0
            transaction_count = 0

            for page in client.iter_transaction_pages(
                account_uid=account["uid"],
                date_from=date_from,
                date_to=date_to,
            ):
                page_count += 1
                transaction_count += len(page["transactions"])

            print(f"Cuenta: {index}")
            print("Saldos recibidos:", len(balances["balances"]))
            print("Páginas de movimientos:", page_count)
            print("Movimientos recibidos:", transaction_count)